# <p style="text-align: center;"> Legal Entity Identifier (LEI) Notebooks </p>
### <p style="text-align: center;">Data Alerts</p>


#### Tasks covered in this notebook:

The notebook provides a simple, practical example of how changes in LEI data can be identified.

* Download the latest **GLEIF Golden Copy** file and the corresponding **delta files**.
* Identify changes in LEI data using GLEIF Golden Copy, API and GoldenCopy Delta Files.
* Randomly select or manually choose **5 LEIs** of interest from the delta files for validation.
* Compare each selected LEI with the data available in the Golden Copy file.
* Determine whether selected LEIs have been changed since their initial issuance. The notebook highlights relevant updates across key data fields, including entity status, legal form, registration details, translated and transliterated names, addresses, and Legal Entity Events.
* Showcase the above steps using the **GLEIF API**, without relying on file downloads.

Golden Copy Delta Files documentation: https://www.gleif.org/en/lei-data/gleif-golden-copy

GLEIF API documentation: https://www.gleif.org/en/lei-data/gleif-api 


In [1]:
# This cell determines whether the notebook is run in Google Colab.

try:
    import google.colab
    IN_COLAB = True

    # Connect your Google Drive to the Colab environment:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust the path:
    import sys
    sys.path.append('/content/drive/MyDrive/lei_notebooks')  # Adjust if necessary

except ImportError:
    IN_COLAB = False


In [2]:
# =============================================================================
# CONFIGURATION OPTIONS
# =============================================================================

# DATA SCOPE OPTIONS
USE_FULL_DATASET = False  # Set to 'False' to use only essential data elements as defined below. 'True' will use all data elements.

# STORAGE OPTIONS
SAVE_TO_DISK = False  # Set to 'False' to keep data only in memory. 'True' will store LEI data and mappings locally

# Delta type: IntraDay | LastDay | LastWeek | LastMonth
DELTA_TYPE = "IntraDay"

# Publication date/time for the Golden Copy baseline
TARGET_DATE = "2026-09-20"
TARGET_TIME = "00:00"  # Golden Copy: 00:00, 08:00, 16:00 UTC

# Shared sample size for both approaches
SAMPLE_SIZE = 20

# API lookback window (days) for field-modifications
API_LOOKBACK_DAYS = 30

# DISPLAY CONFIGURATION
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Data Scope: {'Full Dataset' if USE_FULL_DATASET else 'Essential Columns Only'}")
print(f"Storage: {'Save to Disk' if SAVE_TO_DISK else 'Keep in Memory Only'}")
print(f"Target Date: {TARGET_DATE}  Time: {TARGET_TIME}")
print(f"Delta Type: {DELTA_TYPE}")
print(f"Sample Size: {SAMPLE_SIZE}")
print(f"API Lookback: {API_LOOKBACK_DAYS} days")
print("=" * 60)


CONFIGURATION SUMMARY
Data Scope: Essential Columns Only
Storage: Keep in Memory Only
Target Date: 2026-09-20  Time: 00:00
Delta Type: IntraDay
Sample Size: 20
API Lookback: 30 days


In [3]:
from __future__ import annotations

import random
import re
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

from utils import (
    GoldenCopyDownload,
    GLEIFAPI,
    ColumnNames,
    DataAlertsVisualizer,
    DataHelper
)

IN_COLAB = globals().get("IN_COLAB", False)

REQUIRED_COLUMNS = (
    ColumnNames.ESSENTIAL_COLUMNS
    + ColumnNames.OTHER_NAMES_COLUMNS
    + ColumnNames.TRANSLITERATED_OTHER_NAMES_COLUMNS
    + ColumnNames.LEGAL_ADDRESS_COLUMNS
    + ColumnNames.OTHER_ADDRESS_COLUMNS
    + ColumnNames.TRANSLITERATED_OTHER_ADDRESS_COLUMNS
    + ColumnNames.REGISTRATION_COLUMNS
    + ColumnNames.EVENT_COLUMNS
)

if not USE_FULL_DATASET:
    print(f"Selected Columns: {len(REQUIRED_COLUMNS)} columns")

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")


Selected Columns: 126 columns
Environment: Local


In [4]:
if TARGET_DATE == "latest":
    date = datetime.today().strftime("%Y-%m-%d")
else:
    date = TARGET_DATE

time_preference = TARGET_TIME

DEFAULT_DATA_DIR = Path.cwd()
DEFAULT_GC_DIR = DEFAULT_DATA_DIR / "gc_downloads"

print(f"Using Golden Copy date: {date} {time_preference} UTC")
print(f"Delta type: {DELTA_TYPE}")


Using Golden Copy date: 2026-09-20 00:00 UTC
Delta type: IntraDay


### Download baseline Golden Copy and delta

The Golden Copy Files contain the entire LEI repository. The Golden Copy Delta Files showcase data updates compared to previous Golden Copy publications. The Delta Files are available for the following time intervals:

- Eight hours earlier.
- 24 hours earlier.
- Seven days earlier.
- One month (31 days) earlier.

In [5]:
# Download Golden Copy using the selected setup
level_1_data = await GoldenCopyDownload.download_with_config(
    date=date,
    file_type="lei",
    save_to_disk=SAVE_TO_DISK,
    use_full_dataset=USE_FULL_DATASET,
    essential_columns=REQUIRED_COLUMNS if not USE_FULL_DATASET else None,
    save_dir=str(DEFAULT_GC_DIR),
    time=time_preference,
)


Using subset of 126 columns
Found URL for: https://goldencopy.gleif.org/api/v2/golden-copies/publishes/lei2/20260920-0000.csv
Reading only selected columns: 126 columns
Successfully loaded 3,435,980 rows and 126 columns into memory
Data loaded successfully into memory
Data loaded in memory: 3,435,980 rows × 126 columns
Memory usage: 15784.2 MB


In [6]:
# Download Golden Copy Delta File
delta_data = await GoldenCopyDownload.download_with_config(
    date=date,
    file_type="lei",
    delta=DELTA_TYPE,
    save_to_disk=SAVE_TO_DISK,
    use_full_dataset=USE_FULL_DATASET,
    essential_columns=REQUIRED_COLUMNS if not USE_FULL_DATASET else None,
    save_dir=str(DEFAULT_GC_DIR),
    time=time_preference,
)


Using subset of 126 columns
Found URL for: https://goldencopy.gleif.org/api/v2/golden-copies/publishes/lei2/20260920-0800.csv?delta=IntraDay
Reading only selected columns: 126 columns
Successfully loaded 1,978 rows and 126 columns into memory
Data loaded successfully into memory
Data loaded in memory: 1,978 rows × 126 columns
Memory usage: 9.2 MB


### Display Formatting and Logical Grouping

The below section provides specific helper methods for structuring and presenting LEI change results. It defines the output fields, formats field names for display, and groups detected changes into user-friendly categories such as legal identity, addresses, registration, Legal Entity Events, and relationship information.

In [7]:
CHANGE_COLUMNS = [
    "Logical Group",
    "Field",
    "Previous Value",
    "Current Value",
]

def display_field_name(field: Any) -> str:
    """
    Return only the last part of a field name.

    Examples:
        Entity.LegalAddress.FirstAddressLine -> FirstAddressLine
        lei:FirstAddressLine                 -> FirstAddressLine

    """
    field = str(field or "").strip()

    if not field:
        return ""

    parts = re.split(r"[.:/\\]", field)

    return parts[-1].strip()



def logical_group(field: Any) -> str:
    """Assign Level 1 and Level 2 fields to user-friendly groups."""

    name = DataHelper.field_key(field)

    # Level 2 - Reporting Exceptions

    if any(
        value in name
        for value in [
            "reportingexception",
            "reportingexceptions",
            "exceptioncategory",
            "exceptionreason",
            "exceptionreference",
        ]
    ):
        return "Reporting Exceptions"

    # Level 2 - Parent relationships
    if "direct" in name:
        return "Direct Parent"

    if "ultimate" in name:
        return "Ultimate Parent"
    if any(
        value in name
        for value in [
            "relationship",
            "startnode",
            "endnode",
            "relationshiptype",
            "relationshipstatus",
            "relationshipperiod",
            "relationshipqualifier",
            "relationshipquantifier",
            "accountingstandard",
            "directparent",
            "ultimateparent",
            "parentrelationship",
        ]
    ):
        return "Parent Relationships"

    # Level 1
    if any(
        value in name
        for value in [
            "eventgroup",
            "legalentityevent",
            "legalevent",
            "successor",
            "predecessor",
            "associatedentity",
        ]
    ):
        return "Legal Entity Events (LEE)"

   # Keep Legal Address and Headquarters Address separate.
    if "legaladdress" in name:
        return "Legal Address"

    if "headquartersaddress" in name:
        return "Headquarters Address"

    # Used when the source does not identify the address type.
    if any(
        value in name
        for value in [
            "addressline",
            "postalcode",
            "mailrouting",
            "addressnumber",
            "city",
            "region",
            "country",
        ]
    ):
        return "Address"
        
    # Registration / Validation
    if any(
        value in name
        for value in [
            "registration",
            "managinglou",
            "validationauthority",
            "validatedas",
            "validationdocuments",
            "validationreference",
            "initialregistrationdate",
            "lastupdatedate",
            "nextrenewaldate",
        ]
    ):
        return "Registration / Validation"
        
    # Entity / Legal Identity
    if any(
        value in name
        for value in [
            "legalname",
            "otherentityname",
            "previouslegalname",
            "transliterated",
            "legalform",
            "legaljurisdiction",
            "entitystatus",
            "entitycategory",
            "entitysubcategory",
            "entitycreationdate",
        ]
    ):
        return "Entity / Legal Identity"

    return "Other"


### Console Alert Notification

The below class provides a simple console based notifier that highlights LEIs with relevant changes and displays the affected fields together with their previous and current values.

In [11]:
class ConsoleNotifier():
    """Placeholder notifier for LEIs with relevant changes."""

    def send(self, lei: str, result: dict) -> None:
        status = result.get("status")
        changes = result.get("changes", pd.DataFrame())

        # Alert only when the LEI is new or modified
        if status not in {"MODIFIED", "NEW"}:
            return

        print(f"\n⚠️ ALERT: LEI {lei}")

        if status == "NEW":
            print("New LEI detected.")
            return

        print("Changes detected:")

        if changes is not None and not changes.empty:
            display_columns = [
                column
                for column in [
                    "Logical Group",
                    "Field",
                    "Previous Value",
                    "Current Value",
                ]
                if column in changes.columns
            ]

            print(
                changes[display_columns]
                .to_string(index=False)
            )

### Golden Copy vs Golden Copy Delta File
For each sampled LEI, we compare the delta values to the Golden Copy baseline. New LEIs (that are present only in the delta file) are flagged. Changed fields are shown under logical groups.

The Golden Copy is used as the baseline, while the delta file contains LEIs that may be new or updated.

In [12]:
class GoldenCopyDeltaComparator:
    """Compare Golden Copy data with delta data."""

    def __init__(
        self,
        golden_copy: pd.DataFrame,
        delta: pd.DataFrame,
    ):
        self.baseline_by_lei = DataHelper.index_by_lei(golden_copy)
        self.delta_by_lei = DataHelper.index_by_lei(delta)

        self.common_columns = [
            column
            for column in self.delta_by_lei.columns
            if column in self.baseline_by_lei.columns
        ]

    def sample(
        self,
        sample_size: int = 10,
        seed: Optional[int] = None,
    ) -> List[str]:
        """Randomly select LEIs from the delta data."""
        leis = self.delta_by_lei.index.tolist()

        if not leis:
            raise ValueError(
                "No LEIs are available in the delta data."
            )

        rng = random.Random(seed)

        return rng.sample(
            leis,
            min(sample_size, len(leis)),
        )

    def compare(self, lei: str) -> Dict[str, Any]:
        """Compare one LEI in the delta data against the Golden Copy."""
        lei = DataHelper.normalize_value(lei)

        if lei not in self.delta_by_lei.index:
            return {
                "status": "MISSING",
                "changes": pd.DataFrame(
                    columns=CHANGE_COLUMNS
                ),
            }

        if lei not in self.baseline_by_lei.index:
            return {
                "status": "NEW",
                "changes": pd.DataFrame(
                    columns=CHANGE_COLUMNS
                ),
            }

        previous = self.baseline_by_lei.loc[lei]
        current = self.delta_by_lei.loc[lei]

        changes = []

        for field in self.common_columns:
            previous_value = DataHelper.normalize_value(
                previous.get(field, "")
            )

            current_value = DataHelper.normalize_value(
                current.get(field, "")
            )

            if previous_value == current_value:
                continue

            changes.append(
                {
                    "Logical Group": logical_group(field),
                    "Field": display_field_name(field),
                    "Previous Value": previous_value,
                    "Current Value": current_value,
                }
            )

        changes_df = pd.DataFrame(
            changes,
            columns=CHANGE_COLUMNS,
        )

        status = (
            "MODIFIED"
            if not changes_df.empty
            else "NO_CHANGE"
        )

        return {
            "status": status,
            "changes": changes_df,
        }

This cell prepares the Golden Copy and Delta datasets, and allows users to either provide LEIs of interest or randomly select LEIs from the delta file for comparison.

In [13]:
# Initialize the GoldenCopy Comparator
gc = GoldenCopyDeltaComparator(
    golden_copy=level_1_data,
    delta=delta_data,
)

# get random samples 
# sampled_leis = gc.sample(
#     sample_size=5,
# )

sampled_leis = ['815600F34134B1724F09', '7437004A0WYEZ944KN85', '636700ULHMOHOD83WF59', '2138004TCUYZYBWOKF43', '815600C94ABDECC1A387']

print(
    f"Sampled {len(sampled_leis)} LEIs "
    f"from {len(gc.delta_by_lei):,} delta records:"
)

for lei in sampled_leis:
    print(f"  {lei}")


# get the notification when LEI is in scope
# notifier = ConsoleNotifier()

# for lei in sampled_leis:
#     result = gc.compare(lei)

#     notifier.send(
#         lei,
#         result,
#     )

Sampled 5 LEIs from 1,978 delta records:
  815600F34134B1724F09
  7437004A0WYEZ944KN85
  636700ULHMOHOD83WF59
  2138004TCUYZYBWOKF43
  815600C94ABDECC1A387


In [19]:
# Visualize the results in notebook
file_alerts = DataAlertsVisualizer(
    leis=sampled_leis,
    result_fn=gc.compare,
    title="Golden Copy vs Delta",
    source_label="Golden Copy",
    show_category_filter=True
)

# Set static to 'False' for the interactive view. 'True' to render and preserve a single selected result as static output.
file_alerts.display(static=True)

Field,Previous Value,Current Value
LastUpdateDate,2025-08-30T10:01:03.000+02:00,2026-09-19T10:04:00.000+02:00
NextRenewalDate,2026-10-12T11:36:04.000+02:00,2027-10-12T11:36:04.000+02:00


### Identify data updates using the GLEIF API

The below API class provides an alternative way to identify changes. Instead of directly comparing downloaded files, it retrieves field modifications from the GLEIF API.

It looks for modifications within the configured lookback period, such as the last 30 days, finds the most recent modification date within that period and returns only these changes. Lastly, it displays the changed fields together with their previous and current values.

In [15]:
class APIChangeComparator:
    """Check the latest LEI changes using the GLEIF API."""

    def __init__(
        self,
        api_client: Optional[GLEIFAPI] = None,
        days: int = 30,
        page_limit: int = 200,
        max_pages: int = 5,
    ):
        self.client = api_client or GLEIFAPI()
        self.days = days
        self.page_limit = page_limit
        self.max_pages = max_pages

    @staticmethod
    def _standardize_modifications(
        df: pd.DataFrame,
    ) -> pd.DataFrame:
        """Prepare API modifications for comparison and display."""
        columns = CHANGE_COLUMNS + [
            "modification_date"
        ]

        if df is None or df.empty:
            return pd.DataFrame(columns=columns)

        out = df.copy()

        out = out.rename(
            columns={
                "Previous value": "Previous Value",
                "New value": "Current Value",
            }
        )

        out["modification_date"] = pd.to_datetime(
            out["modification_date"],
            errors="coerce",
            utc=True,
        )

        # Use the full API path to identify the correct group.
        group_source = (
            out["Full field"]
            if "Full field" in out.columns
            else out["field"]
        )

        out["Logical Group"] = (
            group_source
            .map(logical_group)
        )

        # Keep only the final field name for display.
        out["Field"] = (
            out["Field"]
            .map(display_field_name)
        )

        out["Previous Value"] = (
            out["Previous Value"]
            .map(DataHelper.normalize_value)
        )

        out["Current Value"] = (
            out["Current Value"]
            .map(DataHelper.normalize_value)
        )

        return out

    def _latest_modifications(
        self,
        modifications: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Return only changes from the latest modification date
        within the fixed lookback period.
        """
        out = self._standardize_modifications(
            modifications
        )

        if out.empty:
            return out

        cutoff = (
            datetime.now(timezone.utc)
            - timedelta(days=self.days)
        )

        recent = out[
            out["modification_date"].notna()
            & (
                out["modification_date"]
                >= cutoff
            )
        ].copy()

        if recent.empty:
            return recent

        latest_date = (
            recent["modification_date"]
            .dt.date
            .max()
        )

        latest = recent[
            recent["modification_date"].dt.date
            == latest_date
        ]

        return (
            latest
            .sort_values(
                "modification_date",
                ascending=False,
            )
            .reset_index(drop=True)
        )

    def compare(
        self,
        lei: str,
    ) -> Dict[str, Any]:
        """Retrieve the latest API changes for one LEI."""
        lei = DataHelper.normalize_value(lei)

        raw_modifications = (
            self.client.fetch_field_modifications(
                lei=lei,
                page_limit=self.page_limit,
                max_pages=self.max_pages,
            )
        )

        latest = self._latest_modifications(
            raw_modifications
        )

        if latest.empty:
            return {
                "status": "NO_CHANGE",
                "changes": latest,
                "latest_modification_date": None,
                "lookback_days": self.days,
            }

        changes = latest[
            CHANGE_COLUMNS
            + ["modification_date"]
        ].copy()

        return {
            "status": "MODIFIED",
            "changes": changes,
            "latest_modification_date": (
                latest["modification_date"].max()
            ),
            "lookback_days": self.days,
        }

In [18]:
# Initialize the API Comparator
api = APIChangeComparator(
    days=30,
    page_limit=200,
    max_pages=5,
)

api_samples = ['72450090DWS1BRZ8V865', '6367009ZKGYLSF97F136', '2594004ADHAD2VCILF69', '984500IF0A0A6AAC7D93', '636700XC4V4CDZ4B8W04']
# Visualize the results in notebook
api_alerts = DataAlertsVisualizer(
    leis=api_samples,
    result_fn=api.compare,
    title="GLEIF API Changes",
    source_label="GLEIF API",
    show_category_filter=True,
)

# Set static to 'False' for the interactive view. 'True' to render and preserve a single selected result as static output.
api_alerts.display(static=True)

Field,Previous Value,Current Value
PostalCode,1016GD,1015AE
FirstAddressLine,Keizersgracht 452,Singel 126


Field,Previous Value,Current Value
PostalCode,1016GD,1015AE
FirstAddressLine,Keizersgracht 452,Singel 126


Field,Previous Value,Current Value
LastUpdateDate,2025-12-31T07:11:21.135+01:00,2026-09-09T08:32:00.402+02:00


Field,Previous Value,Current Value
ValidationReference,,https://www.kvk.nl/orderstraat/subproduct-kiezen/?kvknummer=08225920&productgroep=Uittreksel
ValidationDocuments,,SUPPORTING_DOCUMENTS
LegalEntityEventRecordedDate,,2026-09-09T08:31:55.404+02:00
LegalEntityEventEffectiveDate,,2026-09-09T08:31:55.404+02:00
LegalEntityEventType,,CHANGE_HQ_ADDRESS
@event_status,,COMPLETED
@group_type,,STANDALONE
LegalEntityEventRecordedDate,,2026-09-09T08:31:55.402+02:00
LegalEntityEventEffectiveDate,,2026-09-09T08:31:55.402+02:00
LegalEntityEventType,,CHANGE_LEGAL_ADDRESS


Version 1.0.0 <br> 
Last updated: 1 October 2026